## Setup

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Imports
import math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
import os

KAGGLE = True
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    print("Running on Kaggle")
else:
    KAGGLE = False
    print("Running Locally")

In [ ]:
MODEL_PATH = 'prajjwal1/bert-tiny'
if KAGGLE:
    model_path = '/kaggle/input/datasets/kamerondawson/bert-tiny-local/bert-tiny-local/'
    # model_path = '/kaggle/input/datasets/kamerondawson/ms-macro-local/ms-marco-local/'
    MODEL_PATH = os.path.abspath(model_path)

## Data

In [ ]:
def max_min_fair_allocation(tok_p, tok_a, tok_b, max_length):
    """
    Distributes capacity fairly among prompt and two responses.
    Uses max-min fairness: satisfy shortest sequence first.
    """
    toks = [('p', tok_p), ('a', tok_a), ('b', tok_b)]
    # Sort by length ascending to satisfy shortest first
    toks.sort(key=lambda x: len(x[1]))
    
    # RoBERTa uses <s> Prompt </s> RespA </s> RespB </s>
    # That is 1 CLS and 3 SEP tokens = 4 special tokens total
    capacity = max_length - 4
    remaining_items = len(toks)
    
    results = {}
    for tid, tok in toks:
        # Fair share is the remaining capacity divided by remaining items
        fair_share = capacity // remaining_items
        alloc = min(len(tok), fair_share)
        
        # results[tid] = tok[:alloc]
        results[tid] = alloc
        capacity -= alloc
        remaining_items -= 1
        
    return results['p'], results['a'], results['b']

In [ ]:
def fix_str(string):
    if string.startswith('["') or string.startswith('[\''):
        string = string[2:]
    elif string.startswith('[') or string.startswith('\''):
        string = string[1:]
    if string.endswith('"]') or string.endswith('\']'):
        string = string[:-2]
    elif string.endswith(']') or string.endswith('\''):
        string = string[:-1]
    return string

In [ ]:
class LlmPreferenceDataset(Dataset):
    def __init__(self, dataframe, is_test=False, model_name=MODEL_PATH, max_length=512):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=KAGGLE)
        self.max_length = max_length
        self.is_test = is_test

        if not self.is_test:
            # Double dataset by swapping A/B to remove positional bias
            orig_features = dataframe[['prompt', 'response_a', 'response_b']].values
            orig_labels = dataframe[['winner_model_a', 'winner_model_b', 'winner_tie']].values

            swapped_features = orig_features.copy()
            swapped_features[:, [1, 2]] = swapped_features[:, [2, 1]]

            swapped_labels = orig_labels.copy()
            swapped_labels[:, [0, 1]] = swapped_labels[:, [1, 0]]

            self.features = np.vstack((orig_features, swapped_features))
            self.labels = np.vstack((orig_labels, swapped_labels))
        else:
            self.features = dataframe[['prompt', 'response_a', 'response_b']].values

    def __len__(self):
        return len(self.features)

    def build_chunks(self, idx, reverse=False):
        """
        Tokenizes prompt + two responses and returns a list of (input_ids, attention_mask)
        tensors — one per sliding window chunk. Variable length across samples; the
        collate_fn handles packing these into a batch.
        """
        raw_prompt, raw_resp_a, raw_resp_b = self.features[idx]
        if reverse:
            raw_resp_a, raw_resp_b = raw_resp_b, raw_resp_a

        prompt  = fix_str(str(raw_prompt))
        resp_a  = fix_str(str(raw_resp_a))
        resp_b  = fix_str(str(raw_resp_b))

        tok_p = self.tokenizer.tokenize(prompt)
        tok_a = self.tokenizer.tokenize(resp_a)
        tok_b = self.tokenizer.tokenize(resp_b)

        # Allocate fair chunk sizes for each piece
        chunk_size_p, chunk_size_a, chunk_size_b = max_min_fair_allocation(
            tok_p, tok_a, tok_b, self.max_length
        )

        # Number of chunks needed so every token of the longest piece is seen
        # Use ceil so the tail is never silently dropped
        def num_chunks(seq_len, chunk_size):
            return math.ceil(seq_len / chunk_size) if chunk_size > 0 else 0

        n_chunks = max(
            num_chunks(len(tok_p), chunk_size_p),
            num_chunks(len(tok_a), chunk_size_a),
            num_chunks(len(tok_b), chunk_size_b),
        )
        n_chunks = max(n_chunks, 1)  # Always at least 1 chunk

        sep = self.tokenizer.sep_token
        cls = self.tokenizer.cls_token

        all_input_ids, all_attention_masks = [], []
        for chunk_idx in range(n_chunks):
            def get_slice(tok, chunk_size):
                if chunk_size == 0 or len(tok) == 0:
                    return []
                if n_chunks == 1:
                    start = 0
                else:
                    # Linearly slide from head to tail so all sequences finish together
                    start = int(round(chunk_idx * (len(tok) - chunk_size) / (n_chunks - 1)))
                    start = max(0, min(start, len(tok) - chunk_size))
                return tok[start : start + chunk_size]

            chunk_p = get_slice(tok_p, chunk_size_p)
            chunk_a = get_slice(tok_a, chunk_size_a)
            chunk_b = get_slice(tok_b, chunk_size_b)

            # RoBERTa format: <s> P </s> A </s> B </s>
            tokens = [cls] + chunk_p + [sep] + chunk_a + [sep] + chunk_b + [sep]
            input_ids = self.tokenizer.convert_tokens_to_ids(tokens)

            pad_len = self.max_length - len(input_ids)
            attention_mask = [1] * len(input_ids) + [0] * pad_len
            input_ids = input_ids + [self.tokenizer.pad_token_id] * pad_len

            all_input_ids.append(torch.tensor(input_ids, dtype=torch.long))
            all_attention_masks.append(torch.tensor(attention_mask, dtype=torch.long))

        # Stack to [n_chunks, max_length]
        return torch.stack(all_input_ids), torch.stack(all_attention_masks)

    def __getitem__(self, idx):
        input_ids, attention_mask = self.build_chunks(idx)
        sample = {
            'input_ids': input_ids,        # [n_chunks, max_length]
            'attention_mask': attention_mask,
        }
        if not self.is_test:
            sample['label'] = torch.tensor(self.labels[idx], dtype=torch.float)
        else:
            ids_sw, mask_sw = self.build_chunks(idx, reverse=True)
            sample['input_ids_swapped']      = ids_sw
            sample['attention_mask_swapped'] = mask_sw
        return sample

In [ ]:
def collate_fn(batch):
    """
    Packs variable-length chunk lists from each sample into a single flat tensor.
    Stores chunk_counts so run_model can split them back out per sample.

    Shapes after collation:
        input_ids      : [total_chunks, max_length]
        attention_mask : [total_chunks, max_length]
        labels         : [batch_size, 3]           (train/val only)
        chunk_counts   : list[int] len=batch_size
    For test samples a parallel set of _swapped keys is included.
    """
    ids_list, mask_list, label_list, counts = [], [], [], []
    ids_sw_list, mask_sw_list, counts_sw = [], [], []
    is_test = 'label' not in batch[0]

    for sample in batch:
        ids_list.append(sample['input_ids'])
        mask_list.append(sample['attention_mask'])
        counts.append(sample['input_ids'].shape[0])
        if is_test:
            ids_sw_list.append(sample['input_ids_swapped'])
            mask_sw_list.append(sample['attention_mask_swapped'])
            counts_sw.append(sample['input_ids_swapped'].shape[0])
        else:
            label_list.append(sample['label'])

    result = {
        'input_ids':      torch.cat(ids_list,  dim=0),   # [total_chunks, seq_len]
        'attention_mask': torch.cat(mask_list, dim=0),
        'chunk_counts':   counts,
    }
    if is_test:
        result['input_ids_swapped']      = torch.cat(ids_sw_list,  dim=0)
        result['attention_mask_swapped'] = torch.cat(mask_sw_list, dim=0)
        result['chunk_counts_swapped']   = counts_sw
    else:
        result['labels'] = torch.stack(label_list)       # [batch_size, 3]
    return result

In [ ]:
if KAGGLE:
    train_set_filepath = '/kaggle/input/llm-classification-finetuning/train.csv'
    test_set_filepath = '/kaggle/input/llm-classification-finetuning/test.csv'
else:
    train_set_filepath = os.path.join(os.getcwd(), 'train.csv')
    test_set_filepath = os.path.join(os.getcwd(), 'test.csv')

# Load datasets
base_train_df = pd.read_csv(train_set_filepath)
if os.path.exists(test_set_filepath):
    base_test_df = pd.read_csv(test_set_filepath)
else:
    base_test_df = None
base_train_df = base_train_df.dropna()
base_test_df = base_test_df.dropna()

# Only the train set has labels so we need to split that into our real train and test sets
train_df, val_df = train_test_split(base_train_df, test_size=0.2, random_state=42)

# Convert to torch datasets
train_set = LlmPreferenceDataset(train_df, is_test=False)
val_set = LlmPreferenceDataset(val_df, is_test=False)
if base_test_df is not None:
    test_set = LlmPreferenceDataset(base_test_df, is_test=True)
else:
    test_set = None

# Create data loaders
BATCH_SIZE = 32
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
if test_set is not None:
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
else:
    test_loader = None

## Model

In [ ]:
class LlmPreferenceModel(nn.Module):
    def __init__(self, model_name=MODEL_PATH, num_classes=3):
        super(LlmPreferenceModel, self).__init__()
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name, 
            num_labels=num_classes,
            local_files_only=KAGGLE,
            ignore_mismatched_sizes=True 
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        return outputs.logits

In [ ]:
DEVICE = torch.device('cpu')
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
print(f"Using device: {DEVICE}")

criterion = nn.CrossEntropyLoss()

# TODO Mix architectures for maximum diversity
architectures = [LlmPreferenceModel] 
models = []
optimizers = []
schedulers = []

for i, ArchClass in enumerate(architectures):
    torch.manual_seed(42 + i)
    model = ArchClass().to(DEVICE)
    models.append(model)
    optimizer = optim.AdamW(model.parameters(), lr=0.00001, weight_decay=0.01)
    optimizers.append(optimizer)
    schedulers.append(optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=1))

# Track accuracy for weighting the final vote
model_scores = [float('inf')] * len(models)

## Train

In [ ]:
def run_model(model, data, reverse=False):
    """
    Runs all chunks for an entire batch in a SINGLE forward pass, then mean-pools
    chunk logits back to one logit vector per sample.

    Args:
        data:    collated batch dict produced by collate_fn
        reverse: if True, use the swapped (TTA) tensors

    Returns:
        Tensor of shape [batch_size, num_classes]
    """
    if reverse:
        flat_ids   = data['input_ids_swapped'].to(DEVICE)       # [total_chunks, seq_len]
        flat_mask  = data['attention_mask_swapped'].to(DEVICE)
        counts     = data['chunk_counts_swapped']
    else:
        flat_ids   = data['input_ids'].to(DEVICE)
        flat_mask  = data['attention_mask'].to(DEVICE)
        counts     = data['chunk_counts']

    # Single forward pass over ALL chunks from ALL samples in the batch
    flat_logits = model(flat_ids, flat_mask)                    # [total_chunks, num_classes]

    # Mean-pool each sample's chunks back into one logit vector
    pooled, start = [], 0
    for n in counts:
        pooled.append(flat_logits[start : start + n].mean(dim=0))
        start += n

    return torch.stack(pooled)                                  # [batch_size, num_classes]

In [ ]:
# Validation
def test_model(model):
    model.eval() # Set model to evaluation mode
    total_log_loss = 0
    total_val_loss = 0.0
    with torch.no_grad():
        for data in val_loader:
            # Run model
            logits = run_model(model, data)
            targets = data['labels'].to(DEVICE)

            # Update loss
            loss = criterion(logits, targets) # Calculate loss
            total_val_loss += loss.item()

            # Update score
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            target_np = targets.cpu().numpy()
            target_indices = np.argmax(target_np, axis=1) 
            total_log_loss += log_loss(target_indices, probs, labels=[0, 1, 2])

    avg_val_loss = total_val_loss / len(val_loader)
    avg_log_loss = total_log_loss / len(val_loader)
    print(f'Leaderboard Log Loss: {avg_log_loss:.4f} | Val Loss: {avg_val_loss:.4f}')
    return avg_val_loss, avg_log_loss

In [ ]:
# Train
EPOCHS = 5
for m_idx, model in enumerate(models):
    print(f"Training Model {m_idx + 1}/{len(models)}...")
    optimizer = optimizers[m_idx]
    scheduler = schedulers[m_idx]

    best_val_loss = float('inf')
    
    for epoch in range(EPOCHS):
        model.train() # Set model back to training mode
        running_loss = 0.0
        for i, data in enumerate(train_loader, 0):
            optimizer.zero_grad()
            logits = run_model(model, data)
            targets = data['labels'].to(DEVICE)
            loss = criterion(logits, targets)
            loss.backward()
            optimizer.step()
    
            running_loss += loss.item()
            if i % 100 == 99: # Print every 100 batches
                print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 100:.3f}')
                running_loss = 0.0
    
        # Run validation and get the loss
        current_val_loss, current_log_loss = test_model(model)
        model_scores[m_idx] = min(model_scores[m_idx], current_log_loss)

        # Save the best weights (Manual Early Stopping)
        if current_val_loss < best_val_loss:
            best_val_loss = current_val_loss
            torch.save(model.state_dict(), 'best_model_weights.pth')
            print(f'New best model saved at Epoch {epoch+1}')
        
        # Step the scheduler based on the validation loss
        scheduler.step(current_val_loss)
        
        current_lr = optimizer.param_groups[0]['lr']
        print(f'End of Epoch {epoch + 1} - Learning Rate: {current_lr}')

    print(f'Finished Training Model {m_idx + 1}. Min Val Log Loss: {model_scores[m_idx]:.2f}%')

print('Finished Training')

## Submission

In [ ]:
# Normalize weights
# Use validation log loss as the 'score'—lower is better, so we use its inverse
inverse_scores = [1.0 / score for score in model_scores]
total_inverse = sum(inverse_scores)
weights = [s / total_inverse for s in inverse_scores]

if test_loader is not None:
    # Match the LMSYS competition format for submission
    submission_data = {'id': [], 'winner_model_a': [], 'winner_model_b': [], 'winner_tie': []}
    
    for m in models: 
        m.eval()
    
    with torch.no_grad():
        for i, data in enumerate(test_loader):
            # Initialize ensemble predictions for this batch
            batch_size = len(data['chunk_counts'])
            ensemble_probs = torch.zeros((batch_size, 3)).to(DEVICE)
            
            # Weighted Average of Probabilities (Soft Voting)
            # It is generally more stable to average probabilities rather than raw logits
            for m_idx, m in enumerate(models):
                # Pass 1: Original order
                logits_1 = run_model(m, data)
                probs_1 = torch.softmax(logits_1, dim=1)
                
                # Pass 2: Swapped Order
                logits_2 = run_model(m, data, reverse=True)
                probs_2 = torch.softmax(logits_2, dim=1)
                
                # Average TTA results for this model
                # Index 0=A, 1=B, 2=Tie. Flip [1, 0, 2] to align B/A back to A/B.
                model_avg_probs = (probs_1 + probs_2[:, [1, 0, 2]]) / 2
                
                # Add weighted contribution to ensemble
                ensemble_probs += model_avg_probs * weights[m_idx]
            
            # Move to CPU for pandas
            ensemble_probs_np = ensemble_probs.cpu().numpy()

            start_idx = i * test_loader.batch_size
            end_idx = start_idx + batch_size
            batch_ids = base_test_df['id'].iloc[start_idx:end_idx].values

            for j, row_id in enumerate(batch_ids):
                submission_data['id'].append(row_id)
                submission_data['winner_model_a'].append(ensemble_probs_np[j, 0])
                submission_data['winner_model_b'].append(ensemble_probs_np[j, 1])
                submission_data['winner_tie'].append(ensemble_probs_np[j, 2])
    
    # Save to CSV
    submission_df = pd.DataFrame(submission_data)
    submission_df.to_csv('submission.csv', index=False)
    print(f"Submission file saved with {len(submission_df)} rows!")

### Verification

In [ ]:
if submission_df is not None:
    # 1. Check for missing values
    missing_values = submission_df.isnull().sum().sum()
    
    # 2. Check for correct column names
    expected_cols = ['id', 'winner_model_a', 'winner_model_b', 'winner_tie']
    missing_cols = [c for c in expected_cols if c not in submission_df.columns]
    
    # 3. Check for valid probability range [0, 1]
    prob_cols = ['winner_model_a', 'winner_model_b', 'winner_tie']
    out_of_range = submission_df[(submission_df[prob_cols] < 0).any(axis=1) | 
                                 (submission_df[prob_cols] > 1).any(axis=1)]
    
    # 4. Check if probabilities sum to ~1.0
    # We use a small epsilon (1e-5) to account for floating point rounding
    sums = submission_df[prob_cols].sum(axis=1)
    invalid_sums = submission_df[~sums.between(0.99, 1.01)]
    
    # 5. Final Verification Report
    print("--- LMSYS Submission Verification ---")
    print(f"Total Rows: {len(submission_df)}")
    print(f"Missing Values: {missing_values}")
    print(f"Missing Required Columns: {len(missing_cols)} ({missing_cols})")
    print(f"Invalid Probabilities (out of 0-1 range): {len(out_of_range)}")
    print(f"Rows where Probs don't sum to 1: {len(invalid_sums)}")
    
    # Validation Logic
    is_count_correct = (len(submission_df) == len(base_test_df))
    is_complete = (missing_values == 0 and len(missing_cols) == 0)
    is_valid = (len(out_of_range) == 0 and len(invalid_sums) == 0)
    
    if is_count_correct and is_complete and is_valid:
        print("\n✅ Verification Passed! Your file is ready for LMSYS submission.")
    else:
        print("\n❌ Verification Failed. Please check the counts above.")
        if not is_count_correct:
            print(f"   - Row count mismatch: Expected {len(base_test_df)}, got {len(submission_df)}")